In [5]:
import pandas as pd

# 读取 clean 数据
d_path = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
d = pd.read_csv(d_path)

# 删除 Cluster 列（如果存在）
if "Cluster" in d.columns:
    d = d.drop(columns=["Cluster"])

# 保存回原文件（覆盖）
d.to_csv(d_path, index=False)

In [6]:
d

,X,Y,gf_Day2017_001_to_gf_Day2017_007,gf_Day2017_002_to_gf_Day2017_008,gf_Day2017_003_to_gf_Day2017_009,gf_Day2017_004_to_gf_Day2017_010,gf_Day2017_005_to_gf_Day2017_011,gf_Day2017_006_to_gf_Day2017_012,gf_Day2017_007_to_gf_Day2017_013,gf_Day2017_008_to_gf_Day2017_014,...,gf_Nit2019_359_to_gf_Nit2019_365,Place,City Name,Place ID,Sum ID,Hemisphere,K_5,K_12,K_20,K_26
0,3.679990e+05,5.108440e+06,0.421664,0.484929,0.635950,0.790895,1.190424,0.211774,0.702198,0.749765,...,-0.701099,0,Lyon [FRA],0,0,1,1,1,5,11
1,3.689514e+05,5.108440e+06,0.807378,0.913501,0.964521,0.962323,1.290424,0.697488,1.145055,1.092622,...,-1.029670,0,Lyon [FRA],1,1,1,1,1,9,11
2,3.699037e+05,5.108440e+06,0.921664,0.984929,1.035950,1.019466,1.304710,0.626060,1.045055,1.021193,...,-0.858242,0,Lyon [FRA],2,2,1,1,1,9,11
3,3.718085e+05,5.108440e+06,0.821664,0.913501,0.978807,0.976609,1.247567,0.297488,0.845055,0.749765,...,0.227473,0,Lyon [FRA],3,3,1,1,1,5,11
4,3.727608e+05,5.108440e+06,-0.092622,-0.029356,0.035950,0.019466,0.090424,-0.959655,-0.554945,-0.664521,...,-0.472527,0,Lyon [FRA],4,4,1,1,1,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,1.131674e+07,2.506661e+06,3.150000,2.971429,2.821429,2.628571,2.764286,2.721429,2.892857,2.850000,...,1.300000,244,"Yulin, Guangxi [CHN]",157,142507,1,2,6,8,18
142502,1.131770e+07,2.505688e+06,2.464286,2.257143,2.107143,1.842857,2.007143,1.907143,1.864286,1.707143,...,0.728571,244,"Yulin, Guangxi [CHN]",158,142508,1,1,1,5,11
142503,1.131962e+07,2.504715e+06,2.507143,2.428571,2.278571,2.028571,2.350000,2.264286,2.250000,2.107143,...,0.428571,244,"Yulin, Guangxi [CHN]",159,142509,1,2,4,4,8
142504,1.132058e+07,2.504715e+06,1.950000,1.814286,1.707143,1.471429,1.935714,1.878571,1.892857,1.778571,...,0.100000,244,"Yulin, Guangxi [CHN]",160,142510,1,2,4,4,8


In [7]:
import pandas as pd

def sort_all_K_clusters(input_file_path, output_file_path):
    # 读取 CSV
    df = pd.read_csv(input_file_path, encoding='gbk')

    # 排除不需要参与计算均值的列
    excluded_columns = ["Place ID", "Sum ID", "City Name", "Place", "Hemisphere", "X", "Y"]

    # 自动识别所有 K 列（假设命名为 K_1, K_2, ...）
    cluster_cols = [col for col in df.columns if col.startswith("K_")]

    print(f"检测到的 Cluster 列: {cluster_cols}")

    for cluster_col in cluster_cols:
        # 选择数值列用于计算均值
        columns_to_average = [col for col in df.columns if col not in excluded_columns + cluster_cols]

        # 按 Cluster 计算均值
        cluster_means = df.groupby(cluster_col)[columns_to_average].mean()

        # 对均值求平均并排序 Cluster
        sorted_clusters = cluster_means.mean(axis=1).sort_values()
        cluster_mapping = {old_cluster: new_cluster for new_cluster, old_cluster in enumerate(sorted_clusters.index)}

        # 更新 Cluster 列
        df[cluster_col] = df[cluster_col].map(cluster_mapping)

        print(f"{cluster_col} 已重新编号")

    # 保存结果
    df.to_csv(output_file_path, index=False, encoding='gbk')
    print(f"已保存排序后的数据到 {output_file_path}")


# 调用示例
cluster_origin_file = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
cluster_processed_file = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
sort_all_K_clusters(cluster_origin_file, cluster_processed_file)



检测到的 Cluster 列: ['K_5', 'K_12', 'K_20', 'K_26']
K_5 已重新编号
K_12 已重新编号
K_20 已重新编号
K_26 已重新编号
已保存排序后的数据到 D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pydlc import dense_lines
import numpy as np

def plot_cluster_data(file_path, K, output_folder):
    data = pd.read_csv(file_path)

    excluded_columns = ["Place ID", "Sum ID", "City Name", "Place", "Hemisphere", "X", "Y"]
    # 自动识别所有 K 列（假设命名为 K_1, K_2, ...）
    cluster_cols = [col for col in data.columns if col.startswith("K_")]

    cluster_col = f'K_{K}'
    if cluster_col not in data.columns:
        print(f"{cluster_col} 不存在")
        return

    cluster_values = data[cluster_col].unique()
    columns_to_average = [col for col in data.columns if col not in excluded_columns + cluster_cols]
    print(f"K={K} 使用的列数量：{len(columns_to_average)}")

    for cluster in cluster_values:
        cluster_data = data[data[cluster_col] == cluster]
        cluster_data_to_plot = cluster_data[columns_to_average]

        # print(f"列名：{columns_to_average}")

        # 统计量
        median = cluster_data_to_plot.median()
        mean = cluster_data_to_plot.mean()
        min_v  = cluster_data_to_plot.min()
        max_v  = cluster_data_to_plot.max()
        q25    = cluster_data_to_plot.quantile(0.25)
        q75    = cluster_data_to_plot.quantile(0.75)

        cols = np.arange(cluster_data_to_plot.shape[1])

        # 创建图
        fig, ax = plt.subplots(figsize=(20, 6))
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')

        # -----------------------------------------
        # ① 画每条折线（全部原始数据）
        # -----------------------------------------
        # for i in range(cluster_data_to_plot.shape[0]):
        #     ax.plot(cols,
        #             cluster_data_to_plot.iloc[i].values,
        #             color="#B95F5F",
        #             alpha=0.05,        # 低透明度不遮挡
        #             linewidth=0.8)


        # 使用 dense_lines 绘制密度线图
        columns = np.arange(cluster_data_to_plot.shape[1])
        im = dense_lines(
                cluster_data_to_plot.values,
                x=columns,
                ax=ax,
                alpha=1,
                cmap='binary'   # 黑白系，更深一些
            )

        # -----------------------------------------
        # ② 中值线
        # -----------------------------------------
        ax.plot(cols, median, '-', color='orange', lw=2.0, label='Median', zorder=10)
        ax.plot(cols, mean, '-', color='Blue', lw=2.0, label='Mean', zorder=10)

        # -----------------------------------------
        # # ③ IQR 25%~75%
        # # -----------------------------------------
        # ax.fill_between(cols, q25, q75,
        #                 color='orange', alpha=0.3, label='25%~75%')

        # -----------------------------------------
        # ④ 0~100% 范围
        # # -----------------------------------------
        # ax.fill_between(cols, min_v, max_v,
        #                 color='orange', alpha=0.15, label='0%~100%')

        # 美化
        ax.set_title(f'sorted_clustered_(K={K})_Cluster {cluster}', fontsize=20)
        ax.set_xlabel('DAY', fontsize=14)
        ax.set_ylabel('slide_TEM', fontsize=14)
        ax.grid(False)
        ax.set_ylim(-20, 20)
        ax.set_xticks(cols[::50])
        ax.set_xticklabels(cols[::50])
        plt.xticks(rotation=90)
        ax.set_xlim(0, len(cols))
        plt.tight_layout()

        # 保存
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, f'sorted_clustered_som_kmeans_data(K={K})_{cluster}.jpg')
        plt.savefig(output_file, dpi=100, bbox_inches='tight')
        plt.close(fig)
# # 循环不同的 K
K_range = [5, 12, 20, 26]

for K in K_range:
    file_path = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
    output_folder = rf'D:\file\d_som\results\picture\1\K={K}'
    os.makedirs(output_folder, exist_ok=True)
    plot_cluster_data(file_path, K, output_folder)


K=5 使用的列数量：2178
K=12 使用的列数量：2178
K=20 使用的列数量：2178
K=26 使用的列数量：2178


In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pydlc import dense_lines
import numpy as np

def plot_cluster_data(file_path, K, output_folder):
    data = pd.read_csv(file_path)

    excluded_columns = ["Place ID", "Sum ID", "City Name", "Place", "Hemisphere", "X", "Y"]
    # 自动识别所有 K 列（假设命名为 K_1, K_2, ...）
    cluster_cols = [col for col in data.columns if col.startswith("K_")]

    cluster_col = f'K_{K}'
    if cluster_col not in data.columns:
        print(f"{cluster_col} 不存在")
        return

    cluster_values = data[cluster_col].unique()
    columns_to_average = [col for col in data.columns if col not in excluded_columns + cluster_cols]

    columns_to_average_day = columns_to_average[0:1089]
    columns_to_average_night = columns_to_average[1089:]
    # print(f"K={K} 使用的列数量：{len(columns_to_average)}")

    for cluster in cluster_values:
        cluster_data = data[data[cluster_col] == cluster]
        cluster_data_to_plot_day = cluster_data[columns_to_average_day]
        cluster_data_to_plot_night = cluster_data[columns_to_average_night]
        print(cluster_data_to_plot_night.columns)

        # 统计量
        mean_day = cluster_data_to_plot_day.mean()
        cols_day = np.arange(cluster_data_to_plot_day.shape[1])
        # 统计量
        mean_night = cluster_data_to_plot_night.mean()
        cols_night = np.arange(cluster_data_to_plot_night.shape[1])

        # 创建图
        fig, ax = plt.subplots(figsize=(20, 6))
        fig.patch.set_facecolor('white')
        ax.set_facecolor('white')

        ax.plot(cols_day, mean_day, '-', color='Red', lw=2.0, label='Mean_Day', zorder=10)
        ax.plot(cols_night, mean_night, '-', color='Blue', lw=2.0, label='Mean_Night', zorder=10)
        # --------------------------
        #  新增：构建 3D 数据表
        # --------------------------
        df_3d = pd.DataFrame({
            "Date": cols_day,          # 时间序列
            "Mean_Day": mean_day.values,
            "Mean_Night": mean_night.values
        })

        print(df_3d.head())

        # 美化
        ax.set_title(f'sorted_clustered_(K={K})_Cluster {cluster}', fontsize=20)
        ax.set_xlabel('DAY', fontsize=14)
        ax.set_ylabel('slide_TEM', fontsize=14)
        ax.grid(False)
        ax.set_ylim(-20, 20)
        ax.set_xticks(cols_day[::30])
        ax.set_xticklabels(cols_day[::30])
        plt.xticks(rotation=90)
        ax.set_xlim(0, len(cols_day))
        plt.tight_layout()

        # 保存
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, f'sorted_clustered_som_kmeans_data(K={K})_{cluster}.jpg')
        plt.savefig(output_file, dpi=100, bbox_inches='tight')
        plt.close(fig)

        output_csv = os.path.join(output_folder, f"K={K}_Cluster{cluster}_3D_Data.csv")
        df_3d.to_csv(output_csv, index=False)
        
K_range = [5, 12, 20, 26]

for K in K_range:
    file_path = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
    output_folder = rf'D:\file\d_som\results\picture\DN\K={K}'
    os.makedirs(output_folder, exist_ok=True)
    plot_cluster_data(file_path, K, output_folder)

Index(['gf_Nit2017_001_to_gf_Nit2017_007', 'gf_Nit2017_002_to_gf_Nit2017_008',
       'gf_Nit2017_003_to_gf_Nit2017_009', 'gf_Nit2017_004_to_gf_Nit2017_010',
       'gf_Nit2017_005_to_gf_Nit2017_011', 'gf_Nit2017_006_to_gf_Nit2017_012',
       'gf_Nit2017_007_to_gf_Nit2017_013', 'gf_Nit2017_008_to_gf_Nit2017_014',
       'gf_Nit2017_009_to_gf_Nit2017_015', 'gf_Nit2017_010_to_gf_Nit2017_016',
       ...
       'gf_Nit2019_350_to_gf_Nit2019_356', 'gf_Nit2019_351_to_gf_Nit2019_357',
       'gf_Nit2019_352_to_gf_Nit2019_358', 'gf_Nit2019_353_to_gf_Nit2019_359',
       'gf_Nit2019_354_to_gf_Nit2019_360', 'gf_Nit2019_355_to_gf_Nit2019_361',
       'gf_Nit2019_356_to_gf_Nit2019_362', 'gf_Nit2019_357_to_gf_Nit2019_363',
       'gf_Nit2019_358_to_gf_Nit2019_364', 'gf_Nit2019_359_to_gf_Nit2019_365'],
      dtype='object', length=1089)
   Date  Mean_Day  Mean_Night
0     0  0.039972    1.029472
1     1  0.059011    1.096938
2     2  0.065239    1.157458
3     3  0.049236    1.127025
4     4  0.02